# 📖 Bible Reading Progress Tracker —Word2Vec + CRF NER Baseline

**Purpose**  
Build a Word2Vec + CRF NER Baseline before moving to IndoBERT.
CRF is interpretable — inspect exactly which features drive predictions

| # | Section | Description |
|---|---------|-------------|
| 1 | **Setup** | Import, paths |
| 2 | **Data** | Load CoNLL, split |
| 3 | **Word2Vec** | Train embeddings on ConLL tokens |
| 4 | **CRF** | Feature engineering (token + Word2Vec) + train + eval |
| 5 | **Inspection** | Top features, transitions, error analysis |
| 6 | **Inference** | Sanity-check pipeline on test sentences |

---
## 1. Setup

In [2]:
import warnings
import sys
import pickle
import numpy as np
from pathlib import Path

import sklearn_crfsuite
from sklearn.model_selection import train_test_split
from seqeval.metrics import classification_report, f1_score
from gensim.models import Word2Vec

warnings.filterwarnings('ignore')
sys.path.append(str(Path('..').resolve() / 'src'))

from config.settings import MODEL_DIR, PROCESSED_DIR

DATA_PATH =PROCESSED_DIR / 'NER_tasks/ner_tasks_1700.conll'
W2V_PATH  = MODEL_DIR / 'word2vec-bible.model'
SAVE_DIR  = MODEL_DIR
SAVE_DIR.mkdir(parents=True, exist_ok=True)

print('Setup complete.')

Setup complete.


## 2. Data

### 2.1 Load CoNLL

CoNLL format - one token per line, blank lines seperate sentences.

```
Efesus  B-BIBLE_REF
1       I-BIBLE_REF
-       I-BIBLE_REF
2       I-BIBLE_REF
done    O
```

In [4]:
def read_conll(filepath: Path):
    """
    Parse a CoNLL-format file into (sentences, labels) lists.
    """
    sentences, labels, raw_texts = [], [], []
    tokens, ner_tags = [], []

    with open(filepath, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()

            if not line:
                if tokens:
                    sentences.append(tokens)
                    labels.append(ner_tags)
                    raw_texts.append(' '.join(tokens))
                    tokens, ner_tags = [], []
            else:
                parts = line.split()
                if parts[0] == "-DOCSTART-":
                    continue

                tokens.append(parts[0])
                ner_tags.append(parts[-1])
    if tokens:
        sentences.append(tokens)
        labels.append(ner_tags)
        raw_texts.append(' '.join(tokens))

    return sentences, labels, raw_texts

In [6]:
import re
import ast
import emoji
import unicodedata
from typing import Any

CROSS_CHAPTER_VERSE_RE = re.compile(
    r'\d+:\d+\s*[-\u2013\u2014]{1,2}\s*\d+:\d+'
    r'|\d+:\d+\s+(?:sampai|sampe|hingga|to)\s+\d+:\d+',
    re.IGNORECASE,
)
VERSE_RANGE_RE = re.compile(
    r'\d+:\d+\s*[-\u2013\u2014]{1,2}\s*\d+(?!\s*:)'
    r'|\d+\s*[-\u2013\u2014]{1,2}\s*\d+:\d+'
    r'|\d+:\d+\s+(?:sampai|sampe|hingga|to)\s+\d+(?!\s*:)',
    re.IGNORECASE,
)
SINGLE_VERSE_RE = re.compile(r'\d+:\d+')
CROSS_BOOK_RE = re.compile(
    r'\d+\s*[-\u2013\u2014]{1,2}\s*[A-Za-z]'
    r'|\d+\s+(?:sampai|sampe|hingga|to)\s+[A-Za-z]',
    re.IGNORECASE,
)
CHAPTER_RANGE_RE = re.compile(
    r'\d+\s*[-\u2013\u2014]{1,2}\s*\d+'
    r'|\d+\s+(?:sampai|sampe|hingga|to)\s+\d+',
    re.IGNORECASE,
)
MARKDOWN_RE = re.compile(r'[_*]')


def is_noisy(msg: str) -> bool:
    has_emoji = any(unicodedata.category(c) in ('So', 'Sm') or c in emoji.EMOJI_DATA for c in msg)
    has_markdown = bool(MARKDOWN_RE.search(msg))
    return has_emoji or has_markdown


def parse_spans(spans: Any) -> list[dict]:
    if isinstance(spans, str):
        return ast.literal_eval(spans)
    return spans or []

def detect_pattern_from_conll(tokens: list[str], ner_tags: list[str]) -> str:
    """Infer pattern pool from token list + BIO labels."""
    text     = ' '.join(tokens)
    ref_toks = [t for t, l in zip(tokens, ner_tags) if l != 'O']
    ref_text = ' '.join(ref_toks)

    if is_noisy(text):
        return 'noisy'

    if not ref_toks:
        return 'none'

    # Count distinct B- spans → multi if ≥ 2
    b_count = sum(1 for l in ner_tags if l == 'B-BIBLE_REF')
    if b_count >= 2:
        return 'multi'

    # Classify the single span by its text
    if CROSS_CHAPTER_VERSE_RE.search(ref_text):
        return 'cross_chapter_verse'
    if VERSE_RANGE_RE.search(ref_text):
        return 'verse_range'
    if SINGLE_VERSE_RE.search(ref_text):
        return 'single_verse'
    if CROSS_BOOK_RE.search(ref_text):
        return 'cross_book'
    if CHAPTER_RANGE_RE.search(ref_text):
        return 'chapter_range'
    if re.search(r'\d', ref_text):
        return 'single_chapter'
    return 'book_only'

In [7]:
sentences, labels, raw_texts = read_conll(DATA_PATH)

pattern_labels = [
    detect_pattern_from_conll(toks, tags)
    for toks, tags in zip(sentences, labels)
]

from collections import Counter
print("Pattern distribution in dataset:")
for pat, n in sorted(Counter(pattern_labels).items()):
    print(f"  {pat:22s}: {n}")

Pattern distribution in dataset:
  book_only             : 127
  chapter_range         : 345
  cross_book            : 276
  cross_chapter_verse   : 17
  multi                 : 279
  noisy                 : 315
  none                  : 106
  single_chapter        : 189
  single_verse          : 8
  verse_range           : 43


### 2.2 Train / Eval Split

80/20 stratified by label presence — ensures both splits see `B-BOOK`, `I-BOOK`, `B-CHAPTER`, `I-CHAPTER` examples.

In [8]:
(train_sent, eval_sent,
 train_labels, eval_labels,
 train_patterns, eval_patterns) = train_test_split(
    sentences, labels, pattern_labels,
    test_size=0.2,
    random_state=42,
    shuffle=True,
    stratify=pattern_labels,  # ← the real stratifier now
)

print(f'Train : {len(train_sent)} | Eval : {len(eval_sent)}')
print(f'\nEval pattern breakdown:')
for pat, n in sorted(Counter(eval_patterns).items()):
    print(f'  {pat:22s}: {n}')

Train : 1364 | Eval : 341

Eval pattern breakdown:
  book_only             : 25
  chapter_range         : 69
  cross_book            : 55
  cross_chapter_verse   : 3
  multi                 : 56
  noisy                 : 63
  none                  : 21
  single_chapter        : 38
  single_verse          : 2
  verse_range           : 9


## 3. Word2Vec Embeddings

Word2Vec learns **dense vector representation of words** based on the contexts in which they appear.
Tokens that occur in similiar contexts tend to have **similar embeddings**.


These embeddings can capture relationships between tokens such as Bible book abbreviations or chapter references.

**Training setup:**
- Corpus is **lowercased** to reduce vocabulary variation
- `vector_size = 50` for a compact embedding space
- `window = 3` to capture nearby context
- `min_count = 1` since the dataset is small
- `epochs = 100` to allow sufficient training

In [9]:
token_corpus = [[t.lower() for t in sent] for sent in sentences]

w2v = Word2Vec(
    sentences=token_corpus,
    vector_size=50,
    window=3,
    min_count=1,
    workers=4,
    epochs=100,
)

w2v.save(str(W2V_PATH))
print(f'Word2Vec trained on {len(w2v.wv)} unique tokens,  dim=50')
print(f'Saved -> {W2V_PATH}')

for query in ['ef', 'kej', 'kor']:
    if query in w2v.wv:
        similar = w2v.wv.most_similar(query, topn=3)
        print(f'  Most similar to {query!r}: {similar}')

Word2Vec trained on 1168 unique tokens,  dim=50
Saved -> C:\one one\Desktop\bible_reading_recap_nlp\models\word2vec-bible.model
  Most similar to 'ef': [('selesai🙏🏻', 0.9155218005180359), ('zed', 0.9005288481712341), ('eat', 0.8842637538909912)]
  Most similar to 'kej': [('50', 0.7319980263710022), ('yes', 0.7176867127418518), ('44', 0.706072986125946)]
  Most similar to 'kor': [('5️⃣', 0.7374691963195801), ('.yudi', 0.7352820038795471), ('2️⃣', 0.7323921918869019)]


---
## 4. CRF

CRF models label sequences by learning:
- **Emission scores** — how likely is label X for this token's features?
- **Transition scores** — how likely is label Y to follow label X?

The key advantage over a simple classifier: it never predicts `I-BOOK` after `B-CHAPTER`.

**Features used:**
- Token shape: lowercase form, prefix/suffix, is_digit, is_title, has_hyphen
- Context: previous and next token features
- Position: beginning/end of sentence
- **Word2Vec**: cosine similarity to known BOOK and CHAPTER seed words

In [10]:
BOOK_SEEDS = [
    'kej', 'kel', 'im', 'bil', 'ul',
    'yos', 'hak', 'rut', 'sam', 'raj',
    'mat', 'mar', 'luk', 'yoh', 'kis',
    'rom', 'kor', 'gal', 'ef', 'kol'
]

CHAPTER_SEEDS = ['1','2','3','4','5','6','7','8','9']

def w2v_similarity(token: str, seeds: list) -> float:
    """Average cosine similarity between a token and a list of seed words."""
    t = token.lower()
    if t not in w2v.wv:
        return 0.0
    sims = [w2v.wv.similarity(t, s) for s in seeds if s in w2v.wv]
    return float(np.mean(sims) if sims else 0.0)

def token_features(tokens, i):
    """Feature dict for token at position i"""
    token = tokens[i]
    t = token.lower()
    
    features = {
        # Token shape
        'token.lower': t,
        'token.is_upper': token.isupper(),
        'token.is_title': token.istitle(),
        'token.is_digit': token.isdigit(),
        'token.has_hyphen': '-' in token,
        'token.has_digit': any(c.isdigit() for c in token),
        'token.prefix2': t[:2],
        'token.prefix3': t[:3],
        'token.suffix2': t[-2:],
        'token.suffix3': t[-3:],
        'token.length': len(token),
        'token.is_first': i == 0,
        'token.is_last': i == len(tokens) - 1,
        # Word2Vec Similarity
        'w2v.book_sim': round(w2v_similarity(t, BOOK_SEEDS), 2),
        'w2v.chapter_sim': round(w2v_similarity(t, CHAPTER_SEEDS), 2),
    }

    if i > 0:
        prev = tokens[i - 1]
        features.update({
            '-1:token.lower': prev.lower(),
            '-1:token.is_title': prev.istitle(),
            '-1:token.is_digit': prev.isdigit(),
            '-1:w2v.book_sim': round(w2v_similarity(prev, CHAPTER_SEEDS), 2),
        })
    else:
        features['BOS'] = True
    
    if i < len(tokens) - 1:
        next = tokens[i + 1]
        features.update({
            '+1:token.lower': next.lower(),
            '+1:token.is_title': next.istitle(),
            '+1:token.is_digit': next.isdigit(),
            '+1:w2v.book_sim': round(w2v_similarity(next, CHAPTER_SEEDS), 2),
        })
    else:
        features['BOS'] = True
    
    return features

def sent_to_features(tokens):
    return [token_features(tokens, i) for i in range(len(tokens))]

In [11]:
# Sanity check
print('Features for first token of first sentence:')
for k, v in token_features(sentences[0], 0).items():
    print(f'  {k:25s}: {v}')

Features for first token of first sentence:
  token.lower              : 14.
  token.is_upper           : False
  token.is_title           : False
  token.is_digit           : False
  token.has_hyphen         : False
  token.has_digit          : True
  token.prefix2            : 14
  token.prefix3            : 14.
  token.suffix2            : 4.
  token.suffix3            : 14.
  token.length             : 3
  token.is_first           : True
  token.is_last            : False
  w2v.book_sim             : 0.31
  w2v.chapter_sim          : 0.25
  BOS                      : True
  +1:token.lower           : inae
  +1:token.is_title        : True
  +1:token.is_digit        : False
  +1:w2v.book_sim          : 0.28


In [12]:
X_train = [sent_to_features(s) for s in train_sent]
y_train = train_labels
X_eval = [sent_to_features(s) for s in eval_sent]
y_eval = eval_labels

print(f'Train: {len(X_train)} sentences, {sum(len(x) for x in X_train)} tokens')
print(f'Eval : {len(X_eval)} sentences,  {sum(len(x) for x in X_eval)} tokens')

Train: 1364 sentences, 7639 tokens
Eval : 341 sentences,  1873 tokens


### 4.1 Train CRF

In [13]:
crf = sklearn_crfsuite.CRF(
    algorithm='lbfgs',
    c1=0.1,
    c2=0.1,
    max_iterations=100,
    all_possible_transitions=True,
)

crf.fit(X_train, y_train)
print('CRF trained.')

CRF trained.


### 4.2 Evaluate CRF

In [14]:
y_pred_crf = crf.predict(X_eval)

print('=== CRF Evaluation ===')
print(classification_report(y_eval, y_pred_crf))
crf_f1 = f1_score(y_eval, y_pred_crf)
print(f'Overall F1: {crf_f1:.4f}')

=== CRF Evaluation ===
              precision    recall  f1-score   support

   BIBLE_REF       0.97      0.96      0.96       393

   micro avg       0.97      0.96      0.96       393
   macro avg       0.97      0.96      0.96       393
weighted avg       0.97      0.96      0.96       393

Overall F1: 0.9641


---
## 5. Inspection

CRF is fully interpretable — examine which features and transitions the model learned.

In [15]:
print('Top features per label:')
for label in ['B-BIBLE_REF', 'I-BIBLE_REF']:
    top = sorted(
        crf.state_features_.items(),
        key=lambda x: -abs(x[1])
    )
    label_features = [(f, w) for (f, l), w in top if l == label][:8]
    print(f'\n  {label}:')
    for feat, weight in label_features:
        print(f'   {feat:35s}: {weight:+.3f}')

Top features per label:

  B-BIBLE_REF:
   +1:token.lower:kor                 : +2.809
   w2v.book_sim                       : +2.762
   token.is_last                      : -2.567
   -1:token.lower:20-21-1             : +2.207
   -1:token.lower:20-21-kisah         : +2.152
   -1:token.lower:done                : +2.018
   +1:token.lower:selesai             : +1.970
   -1:token.lower:kitab               : +1.963

  I-BIBLE_REF:
   token.is_first                     : -3.478
   -1:token.lower:-                   : +3.149
   token.has_digit                    : +2.981
   token.has_hyphen                   : +2.873
   BOS                                : -2.533
   +1:token.lower:kor                 : -2.120
   token.suffix2:️⃣                   : -1.885
   +1:token.is_title                  : -1.395


In [16]:
print('Top transition weights:')
top_transitions = sorted(
    crf.transition_features_.items(),
    key=lambda x: -x[1]
)[:10]
for (from_label, to_label), weight in top_transitions:
    print(f'  {from_label:12s} → {to_label:12s}: {weight:+.3f}')

Top transition weights:
  O            → O           : +5.061
  B-BIBLE_REF  → I-BIBLE_REF : +2.307
  O            → B-BIBLE_REF : +1.606
  I-BIBLE_REF  → I-BIBLE_REF : +0.587
  I-BIBLE_REF  → O           : -0.144
  B-BIBLE_REF  → O           : -0.740
  I-BIBLE_REF  → B-BIBLE_REF : -3.090
  B-BIBLE_REF  → B-BIBLE_REF : -3.511
  O            → I-BIBLE_REF : -3.752


In [17]:
errors = []
for tokens, true_seq, pred_seq in zip(eval_sent, y_eval, crf.predict(X_eval)):
    for token, true_label, pred_label in zip(tokens, true_seq, pred_seq):
        if true_label != pred_label:
            errors.append({
                'token': token,
                'true_label': true_label,
                'pred_label': pred_label,
            })

print(f'Total errors : {len(errors)}\n')
print(f'{"Token":20s} {"True":15s} {"Predicted":15s}')
print('-' * 52)
for e in errors[:20]:
    print(f'{e["token"]:20s} {e["true_label"]:15s} {e["pred_label"]:15s}')

Total errors : 27

Token                True            Predicted      
----------------------------------------------------
Baca                 O               B-BIBLE_REF    
10                   O               I-BIBLE_REF    
Bil31                B-BIBLE_REF     O              
dan                  B-BIBLE_REF     O              
-                    O               I-BIBLE_REF    
Dan                  B-BIBLE_REF     I-BIBLE_REF    
29.Riri              O               B-BIBLE_REF    
2                    B-BIBLE_REF     I-BIBLE_REF    
2                    B-BIBLE_REF     I-BIBLE_REF    
Mark                 B-BIBLE_REF     O              
10:45                I-BIBLE_REF     O              
Yohanes              O               B-BIBLE_REF    
belum                O               I-BIBLE_REF    
Markus               O               B-BIBLE_REF    
yojanes              B-BIBLE_REF     O              
Micah                B-BIBLE_REF     O              
6:8                  I-BIBL

In [18]:
crf_save_path = SAVE_DIR / 'crf-bible-ner.pkl'
with open(crf_save_path, 'wb') as f:
    pickle.dump(crf, f)
print(f'CRF saved → {crf_save_path}')

CRF saved → C:\one one\Desktop\bible_reading_recap_nlp\models\crf-bible-ner.pkl


---
## 6. Inference

Run CRF on the standard test sentences.

In [19]:
import joblib

crf = joblib.load(SAVE_DIR / 'crf-bible-ner.pkl')
w2v = Word2Vec.load(str(SAVE_DIR / 'word2vec-bible.model'))

In [20]:
test_sentences = [
    # Standard patterns 
    'Efesus 1-2 done',
    'Why 1-2 done',
    'Efs1-2 done',                        
    '_Ef 1-2_ ✓',                          
    'Gal 3-6 done',
    '2 Kor 4-9🙏',
    '2 Pet 11-13 ☑️',
    '1 Korintus 14 - 15 selesai.',
    'Kis 1-10 done',

    # Cross-book ranges
    '1 Korintus 16 - 2 Korintus 1 selesai',
    'Gal 6- Ef 2 done',
    'Ibrani 11-Yakobus 1; 2-3; 4-5',
    'Kel 40-Im 2 done',
    'Bil 36 - Ul 2 selesai',

    # Comma / list patterns 
    '1 Kor 1,2,3,4,5 done',
    'Mazmur 1, 23, 91 selesai',
    'Mat 5, 6, 7 done',
    'Kej 1,3,5,7 selesai',

    # Minor Prophets comma list 
    'Amos, Obaja, Yunus, Mikha, Nahum, Habakuk, Zefanya, Hagai, Zakaria, Maleaki done',

    # Multiline 
    'Kej 1-3 done\nKej 4-6 done\nKej 7-9 done',
    'Dan 1-2 done\nDan 3-4 done',
    '3 Yohanes ✅\nYudas ✅',

    # Compound book names 
    'hakim-hakim 6 - hakim-hakim 7 selesai',
    '```Kidung Agung 5 - Kidung Agung 6```✅',
    '1 Raja-raja 3-5 done',

    # Book-range prefix
    '1-3 Yoh done',
    '1-2 Kor done',
    '1-2 Samuel selesai',

    # Semi-colon separated
    '1 petrus done; 2 petrus done; 3 petrus done',
    'Kej 1-10 done; Kel 1-5 done',

    # Verse-level / mixed formats
    'I Sam 2:1-5',
    'Mat 6:9-13 selesai',
    'Yoh 3:16',
    'Roma 8:28 done',

    # Noisy / real-life progress text
    "17. Jason done kej 1-30",
    'Saya sudah sampai Kej 36, terima kasih',
    'Bu Dessi, saya belum selesai baca Daniel 13',
    'Progress hari ini: Kel 1-12 selesai 🙏',

    # Hard negatives / ambiguity
    'Besok daniel ulang tahun',
    'Jojo daniel 1-3 done',
    'Zed 1-3 done',
    'Raja film 2 bagus banget',

    # Unicode / dash variations
    'Gal 6—Ef 2 done',
    'Ef 1–3 selesai',
    'Kej 1—3 done',
    'Kej 1 s/d 2 done',
    ' Mikha 6/7 done',

    # Messy / OCR-like / spacing issues
    '2 Raja - blraja 6 - 7 done',
    '1kor1-3 done',
    'kej1 - 5done',
    'Maz  23  done',

    # Long chained sequences
    '1 Raja-raja 20-21 done 1 Raja-raja 22 - 2 Raja-raja 1 done 2 Raja-raja 2-3 done',
    
    # Book only
    'Mazmur done',
    'Kejadian selesai',
    'Wahyu ✓',
    "1 Raja dan 2 Raja",
]

def crf_predict(sentence: str) -> list:
    """Run CRF inference on a raw sentence string."""
    tokens = sentence.split()
    feats = sent_to_features(tokens)
    preds = crf.predict([feats])[0]
    return list(zip(tokens, preds))

print('=== CRF Inference ===')
for sentence in test_sentences:
    result = crf_predict(sentence)
    entities = [(t, l) for t, l in result if l != 'O']
    print(f'   {sentence!r:45s} → {entities}')

=== CRF Inference ===
   'Efesus 1-2 done'                             → [('Efesus', 'B-BIBLE_REF'), ('1-2', 'I-BIBLE_REF')]
   'Why 1-2 done'                                → [('Why', 'B-BIBLE_REF'), ('1-2', 'I-BIBLE_REF')]
   'Efs1-2 done'                                 → [('Efs1-2', 'B-BIBLE_REF')]
   '_Ef 1-2_ ✓'                                  → [('_Ef', 'B-BIBLE_REF'), ('1-2_', 'I-BIBLE_REF')]
   'Gal 3-6 done'                                → [('Gal', 'B-BIBLE_REF'), ('3-6', 'I-BIBLE_REF')]
   '2 Kor 4-9🙏'                                  → [('2', 'B-BIBLE_REF'), ('Kor', 'I-BIBLE_REF'), ('4-9🙏', 'I-BIBLE_REF')]
   '2 Pet 11-13 ☑️'                              → [('2', 'B-BIBLE_REF'), ('Pet', 'I-BIBLE_REF'), ('11-13', 'I-BIBLE_REF')]
   '1 Korintus 14 - 15 selesai.'                 → [('1', 'B-BIBLE_REF'), ('Korintus', 'I-BIBLE_REF'), ('14', 'I-BIBLE_REF'), ('-', 'I-BIBLE_REF'), ('15', 'I-BIBLE_REF')]
   'Kis 1-10 done'                               → [('Kis', 'B-BIBLE_REF'), (